# BraTS Segmentation Mask Preprocessing

Aligns the BraTS 2023 GLI segmentation masks with the T1N volumes from
`brats_processing.ipynb`. The crop bounding box is computed from the T1N
volume and re-used for the seg mask so the two stay spatially aligned.
Resizing uses nearest-neighbor interpolation to preserve the discrete labels.

Labels: 0 = background, 1 = NCR, 2 = edema, 3 = enhancing tumor.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd()))

from src.nifti import load_nifti, find_seg_files
from src.preprocessing import preprocess_seg, process_brats_seg_dataset


PRESETS = {
    16: (16, 16, 16),
    32: (32, 32, 32),
    48: (48, 48, 48),
    64: (64, 64, 64),
}

PRESET = 32
TARGET_SIZE = PRESETS[PRESET]

INPUT_DIR  = "/home/sven/Desktop/diffsuion/brain"
OUTPUT_DIR = f"/home/sven/Desktop/diffsuion/Processed/brats/{PRESET}_seg"

print(f"preset {PRESET} -> target size {TARGET_SIZE}")
print(f"input:  {INPUT_DIR}")
print(f"output: {OUTPUT_DIR}")


In [ ]:
pairs = find_seg_files(INPUT_DIR)
print(f"found {len(pairs)} (T1N, seg) pairs")

subject_id, t1n_path, seg_path = pairs[0]
t1n_raw, _ = load_nifti(t1n_path)
seg_raw, _ = load_nifti(seg_path)

print(f"\nsample: {subject_id}")
print(f"t1n shape {t1n_raw.shape}, range [{t1n_raw.min():.0f}, {t1n_raw.max():.0f}]")
print(f"seg shape {seg_raw.shape}, labels {np.unique(seg_raw).astype(int).tolist()}")

d = t1n_raw.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(t1n_raw[d], cmap="gray")
axes[0].set_title("t1n axial")
axes[0].axis("off")
axes[1].imshow(seg_raw[d], cmap="nipy_spectral", vmin=0, vmax=3)
axes[1].set_title("seg axial")
axes[1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
processed = preprocess_seg(t1n_raw, seg_raw, TARGET_SIZE)
print(f"processed shape {processed.shape}, labels {np.unique(processed).astype(int).tolist()}")

d, h, w = processed.shape
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(processed[d // 2], cmap="nipy_spectral", vmin=0, vmax=3)
axes[0].set_title("axial")
axes[1].imshow(processed[:, h // 2], cmap="nipy_spectral", vmin=0, vmax=3)
axes[1].set_title("coronal")
axes[2].imshow(processed[:, :, w // 2], cmap="nipy_spectral", vmin=0, vmax=3)
axes[2].set_title("sagittal")
for ax in axes:
    ax.axis("off")
plt.suptitle(f"{subject_id} processed seg ({TARGET_SIZE[0]}^3)")
plt.tight_layout()
plt.show()


In [ ]:
written = process_brats_seg_dataset(INPUT_DIR, OUTPUT_DIR, TARGET_SIZE)


In [ ]:
out_path = written[0]
seg_out, _ = load_nifti(out_path)
print(f"{Path(out_path).name}: shape={seg_out.shape}, "
      f"labels={np.unique(seg_out).astype(int).tolist()}")

d, h, w = seg_out.shape
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(seg_out[d // 2], cmap="nipy_spectral", vmin=0, vmax=3)
axes[0].set_title("axial")
axes[1].imshow(seg_out[:, h // 2], cmap="nipy_spectral", vmin=0, vmax=3)
axes[1].set_title("coronal")
axes[2].imshow(seg_out[:, :, w // 2], cmap="nipy_spectral", vmin=0, vmax=3)
axes[2].set_title("sagittal")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
